# Prompt Templates 与 Output Parsers

对标文档：LangChain Core - Prompt Templates / Structured Output

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

### 1. ChatPromptTemplate 带变量

In [ ]:
prompt = ChatPromptTemplate.from_template(
    "你是一个{role}专家，请回答：{question}"
)
chain = prompt | llm | StrOutputParser()

print(chain.invoke({"role": "Python", "question": "什么是装饰器？"})[:200])

### 2. 结构化输出（Pydantic）

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field

class MovieReview(BaseModel):
    title: str = Field(description="电影名称")
    rating: float = Field(description="评分 1-10")
    summary: str = Field(description="一句话短评")

parser = PydanticOutputParser(pydantic_object=MovieReview)

review_prompt = ChatPromptTemplate.from_messages([
    ("system", "提取结构化数据。{format_instructions}"),
    ("human", "{input}"),
]).partial(format_instructions=parser.get_format_instructions())

chain = review_prompt | llm | parser
review = chain.invoke({"input": "《黑客帝国》科幻片，基努·里维斯主演，评分9分。"})
print(f"电影: {review.title}")
print(f"评分: {review.rating}")
print(f"短评: {review.summary}")